## Görev 1

In [1]:
# Tek head'i multi-head attention'a çevir: aynı boyutu n_head tane küçük head'e böl, çıktıları birleştir. Val loss'u geçen haftaki tek head'le yan yana koy. (1:21:59 - 1:24:25)

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-10-09 10:47:14--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.03s   

2026-10-09 10:47:14 (33.8 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [5]:
# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
# ------------

In [6]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [7]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]  # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [8]:
# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [10]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([h(x) for h in self.heads], dim=-1)   # sadece birleştir, henüz proj yok

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)   # n_head tane küçük head, birleşince n_embd'ye tamamlanıyor
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_heads(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.022721 M parameters


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.1777, val loss 4.1801
step 100: train loss 3.0078, val loss 3.0326
step 200: train loss 2.7707, val loss 2.7792
step 300: train loss 2.6848, val loss 2.6938
step 400: train loss 2.6310, val loss 2.6325
step 500: train loss 2.5849, val loss 2.5928
step 600: train loss 2.5454, val loss 2.5635
step 700: train loss 2.5184, val loss 2.5277
step 800: train loss 2.4955, val loss 2.4902
step 900: train loss 2.4614, val loss 2.4711
step 1000: train loss 2.4391, val loss 2.4553
step 1100: train loss 2.4261, val loss 2.4212
step 1200: train loss 2.4105, val loss 2.4110
step 1300: train loss 2.3833, val loss 2.3980
step 1400: train loss 2.3673, val loss 2.3899
step 1500: train loss 2.3589, val loss 2.3710
step 1600: train loss 2.3347, val loss 2.3547
step 1700: train loss 2.3328, val loss 2.3387
step 1800: train loss 2.3228, val loss 2.3404
step 1900: train loss 2.3082, val loss 2.3261
step 2000: train loss 2.2987, val loss 2.3168
step 2100: train loss 2.2965, val loss 2.3147


In [ ]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


that my tece start
hien bot dall, I hourf my le have houring link.
Then yoy Thang wiell thay,-
The st din.
Burpbee
Therront soedve thed mand, a theve parchinge ast.
Whou Hithir ane wish!
Covenll phick, oumhes fore of anower'th us mipheerther kela! 't, Good ean!

I the 'wy our lordne seocesings, Whiccone ha me; an ther toose poy
And mant mostesay will purn tosueds,
And whes wispseits come sur and, na I I
Lood boyou he not weargomy loce berewith
Tot shin lovee, 'th
For as suwe hit the his eare hes oup,

Whit thid stinge ve cump teand, meip,
Noot with! Chear sand?

Thos lis marmest, Ed:
Wered abe I
may, o btiand ghe rend thee urspe war kremenk.

MONIUS:
Thipere sow, Nu.

RimYUO:
Five a lon; te bold:
Tave, herry for lia
You withen rrien:
And ghatlim: ris meagrie us surne the gat shir car:
I beacole!
She gione the baiend; ares ingtaul now beet
SyUS mity hir Ton speaccant a in lyou wor;
Uthrso wale my.

That sheades tiand oscan? therre rend sought ink, burd,
Time of hang them feive No I an 

In [ ]:
# Tek Headli val loss: 2.3795
# 4 Headli val loss: 2.1999

## Görev 2

In [ ]:
# Transformer bloğunu parça parça tamamla: feedforward, residual bağlantı, LayerNorm. Her parçayı ekledikten sonra val loss'u kaydet. Tek tablo çıkar: bigram, tek head, multi-head, feedforward eklenmiş, residual eklenmiş, LayerNorm eklenmiş. LayerNorm'u hafta 6'da yazdığın BatchNorm1d ile karşılaştır: hangi eksende normalize ediyor, neden running_mean tutmuyor, kendi cümlelerinle yaz. (1:24:25 - 1:37:49)

In [ ]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_heads(x)
        x = self.ffwd(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(estimate_loss())

0.055809 M parameters
step 0: train loss 4.1784, val loss 4.1783
step 100: train loss 2.9159, val loss 2.9327
step 200: train loss 2.6687, val loss 2.6844
step 300: train loss 2.5919, val loss 2.5933
step 400: train loss 2.5309, val loss 2.5346
step 500: train loss 2.4671, val loss 2.4661
step 600: train loss 2.4304, val loss 2.4284
step 700: train loss 2.3859, val loss 2.3777
step 800: train loss 2.3472, val loss 2.3542
step 900: train loss 2.3081, val loss 2.3181
step 1000: train loss 2.3010, val loss 2.3028
step 1100: train loss 2.2549, val loss 2.2707
step 1200: train loss 2.2277, val loss 2.2454
step 1300: train loss 2.2216, val loss 2.2341
step 1400: train loss 2.1890, val loss 2.2037
step 1500: train loss 2.1655, val loss 2.1958
step 1600: train loss 2.1501, val loss 2.1847
step 1700: train loss 2.1220, val loss 2.1639
step 1800: train loss 2.1072, val loss 2.1437
step 1900: train loss 2.0967, val loss 2.1427
step 2000: train loss 2.0794, val loss 2.1340
step 2100: train loss 2.

In [ ]:
# FeedForward val loss çıktısı: 1.9923

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = x + self.sa_heads(x)   # residual: attention çıktısı girdiye eklendi
        x = x + self.ffwd(x)       # residual: feedforward çıktısı da eklendi
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(estimate_loss())

0.059969 M parameters
step 0: train loss 4.5742, val loss 4.5673
step 100: train loss 2.7051, val loss 2.7132
step 200: train loss 2.5649, val loss 2.5630
step 300: train loss 2.4833, val loss 2.4965
step 400: train loss 2.4213, val loss 2.4186
step 500: train loss 2.3631, val loss 2.3683
step 600: train loss 2.3119, val loss 2.3242
step 700: train loss 2.2849, val loss 2.2935
step 800: train loss 2.2654, val loss 2.2694
step 900: train loss 2.2263, val loss 2.2523
step 1000: train loss 2.2103, val loss 2.2302
step 1100: train loss 2.1781, val loss 2.2088
step 1200: train loss 2.1534, val loss 2.1783
step 1300: train loss 2.1406, val loss 2.1845
step 1400: train loss 2.1129, val loss 2.1608
step 1500: train loss 2.0984, val loss 2.1416
step 1600: train loss 2.0870, val loss 2.1360
step 1700: train loss 2.0701, val loss 2.1259
step 1800: train loss 2.0576, val loss 2.1077
step 1900: train loss 2.0349, val loss 2.1172
step 2000: train loss 2.0262, val loss 2.1020
step 2100: train loss 2.

In [ ]:
# residual val loss çıktısı: 1.9623

In [ ]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedFoward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = x + self.sa_heads(x)
        x = x + self.ffwd(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(estimate_loss())

0.059969 M parameters
step 0: train loss 4.5077, val loss 4.5130
step 100: train loss 2.7080, val loss 2.7120
step 200: train loss 2.5606, val loss 2.5733
step 300: train loss 2.4763, val loss 2.4932
step 400: train loss 2.4171, val loss 2.4190
step 500: train loss 2.3573, val loss 2.3667
step 600: train loss 2.3187, val loss 2.3358
step 700: train loss 2.2781, val loss 2.2946
step 800: train loss 2.2475, val loss 2.2693
step 900: train loss 2.2187, val loss 2.2499
step 1000: train loss 2.1956, val loss 2.2271
step 1100: train loss 2.1695, val loss 2.2003
step 1200: train loss 2.1487, val loss 2.1862
step 1300: train loss 2.1388, val loss 2.1780
step 1400: train loss 2.1304, val loss 2.1677
step 1500: train loss 2.0986, val loss 2.1487
step 1600: train loss 2.0863, val loss 2.1344
step 1700: train loss 2.0805, val loss 2.1175
step 1800: train loss 2.0622, val loss 2.1301
step 1900: train loss 2.0510, val loss 2.1219
step 2000: train loss 2.0530, val loss 2.0918
step 2100: train loss 2.

In [ ]:
# LayerNorm val loss çıktı: 1.9509

In [ ]:
import pandas as pd

data = {
    "Model": ["Bigram (context 8)", "Tek head (context 8)", "Multi-head, 4 head (context 32)",
              "+ FeedForward", "+ Residual", "+ LayerNorm"],
    "Parametre Sayısı": [4225, 7553, 22721, 55809, 59969, 59969],
    "Val Loss": [2.4793, 2.3795, 2.1999, 1.9923, 1.9623, 1.9509],
}

df = pd.DataFrame(data)
df

,Model,Parametre Sayısı,Val Loss
0,Bigram (context 8),4225,2.4793
1,Tek head (context 8),7553,2.3795
2,"Multi-head, 4 head (context 32)",22721,2.1999
3,+ FeedForward,55809,1.9923
4,+ Residual,59969,1.9623
5,+ LayerNorm,59969,1.9509


In [ ]:
# Her eklediğimiz parça loss'u ayrı ayrı düşürdü. En büyük düşüş FeedForward eklendiğinde oldu,
# çünkü attention sadece hangi tokena ne kadar bakması gerektiğen kendisi karar veriyordu ancak  gerçek düşünme/işleme işini FeedForward yapıyor.
# Residual ve LayerNorm'un katkısı daha küçük kaldı ama onlar da loss'u olumlu yönde düşürdü.
# Residual ve layernormun asıl faydası tek katmanlı bir modelde değil, katman sayısı (n_layer) arttığında ortaya çıkacak.
# residual bağlantılar olmazsa derin ağlarda gradyan kaybolur, LayerNorm da katmanlar arasında
# aktivasyonları stabil tutar. Yani burada küçük görünen fayda, model büyüdükçe çok daha kritik hale gelir.

# BatchNorm1d: sütun bazlı çalışır bir sütunu alır ve  o sütundaki tüm satırların
# batch'teki 32 örnek üzerindeki ortalamasını alıyor.
# LayerNorm: satır bazlı çalışır  bir satırı alır ve  o satırdaki tüm sütunların
# her örneğin, kendi sayısı üzerindeki ortalamasını alıyor.

# BatchNorm, inference sırasında tek satır geldiğinde sütun boyunca ortalama
# alamaz, çünkü ortalayacak başka satır yok. Bu yüzden running_mean ve running_var'ı
# eğitim boyunca biriktirmek zorundaydı. LayerNorm ise zaten tek bir satırın kendi içinde
# çalıştığı için, tek örnek gelse de hesaplayacağı şey o satırın kendi ortalaması olduğu için hep elinde
# hazır tutar bu yüzden running_mean veya running_var tutmasına hiç gerek yok.

## Görev 3

In [ ]:
# Modeli büyüt ve dropout ekle. Elindeki donanıma sığan bir konfigürasyon seç: n_embd, n_head, n_layer, block_size. Parametre sayısını, val loss'u ve eğitim süresini yaz, neden bu değerleri seçtiğini açıkla. Modelin ürettiği metinden bir örnek koy. (1:37:49 - 1:42:39)

In [16]:
batch_size = 64
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 4
dropout = 0.2

In [17]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

0.824897 M parameters


In [18]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
elapsed = time.time() - start_time
print(f"Eğitim süresi: {elapsed/60:.2f} dakika")

print(estimate_loss())

step 0: train loss 4.3392, val loss 4.3288
step 500: train loss 2.3846, val loss 2.3958
step 1000: train loss 2.1193, val loss 2.1641
step 1500: train loss 1.9237, val loss 2.0063
step 2000: train loss 1.7958, val loss 1.9195
step 2500: train loss 1.7126, val loss 1.8604
step 3000: train loss 1.6548, val loss 1.8174
step 3500: train loss 1.6037, val loss 1.7816
step 4000: train loss 1.5694, val loss 1.7535
step 4500: train loss 1.5417, val loss 1.7286
step 4999: train loss 1.5193, val loss 1.7097
Eğitim süresi: 3.91 dakika
{'train': tensor(1.5189), 'val': tensor(1.7065)}


In [14]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


LEONTES:
And furiolow that all will not, villing
Mest dame up that discorpice, my come!

MARGARET:
Now, him on the them; by all that palent
I am his for thou arrity; and so glore do this good
Thinks thus sheeks; why, doth'st I canne the rave are
Toster'd
Alas to moshour of jince; as not come: being to aughd,
Ford thy you still have and sate,
Where Intisely deam too forthence, vil ip on senut
The pup that of should a
tis: sir; that's chargming naw a ferewel noin?

DUKE VINCENTIO:
Ttake is divisha


In [ ]:
# Neden bu değerleri seçtim:
# T4 GPU'ya sığacak ama birkaç dakikada eğitilebilecek bir denge aradım.
# n_embd'yi 64'ten 128'e, block_size'ı 32'den 128'e çıkararak hem her tokenı
# hem de modelin görebildiği bağlamı büyüttüm. n_layer=4 ile modele birden fazla
# attention+feedforward katmanı üst üste ekleyip daha derin bir yapı kurdum.
# Bu konfigürasyon toplam 0.824897M parametre verdi ve T4'te 5000 adımı sadece 3.72
# dakikada tamamladı.
#
# Sonuç: val loss 1.9509'dan 1.6966'ya düştü. Bu, bir önceki görevdeki tek katmanlı
# modelin (feedforward+residual+LayerNorm eklenmiş ama n_layer=1, n_embd=64) üzerine,
# hem katman sayısını hem de genişliği artırmanın gerçekten işe yaradığını gösteriyor.

In [19]:
# başka hiperparametrelerle bir daha deneyelim
batch_size = 64
block_size = 128
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 192
n_head = 6
n_layer = 6
dropout = 0.2

In [20]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

2.715713 M parameters


In [21]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
elapsed = time.time() - start_time
print(f"Eğitim süresi: {elapsed/60:.2f} dakika")

print(estimate_loss())

step 0: train loss 4.3165, val loss 4.3136
step 500: train loss 2.2198, val loss 2.2470
step 1000: train loss 1.8816, val loss 1.9790
step 1500: train loss 1.6895, val loss 1.8364
step 2000: train loss 1.5820, val loss 1.7535
step 2500: train loss 1.5125, val loss 1.6988
step 3000: train loss 1.4580, val loss 1.6508
step 3500: train loss 1.4180, val loss 1.6187
step 4000: train loss 1.3872, val loss 1.5903
step 4500: train loss 1.3583, val loss 1.5716
step 4999: train loss 1.3391, val loss 1.5590
Eğitim süresi: 10.05 dakika
{'train': tensor(1.3406), 'val': tensor(1.5585)}


In [22]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))


GLOUCESTER:
But is sunsecution the man eye reposity.

AINABELLO:
Nay, Go: a noing-believe distruct, and her in
that a fit, therefore woudd be rismindlam'd to the pake in out
gry when twere he's me; we ho!
Thou
He is be with it dearl there's paitor;
On he in merrettinutents,
Of whitestimes Has laughtery shall be thints,
Loth heaven they bids to had more o'er-un hisnonours
Manio. That Off wall undert'sument
Blend be, who your head not did it his consun,
The man? ill is a Verian being lifuds not be


## Görev 4

In [ ]:
# Aynı modeli Türkçe metinle eğit. Metni kendin derle, Tiny Shakespeare'e yakın boyutta olsun (yaklaşık 1 MB). Kaynaklardaki Vikikaynak'tan başlayabilirsin. Ürettiği metinden örnek ver. Türkçe metnin kaç farklı karakteri var, Shakespeare'inkiyle karşılaştır. İki val loss'u doğrudan karşılaştırabilir misin, neden, yaz.

In [29]:
import requests
import re
import time

API = "https://tr.wikisource.org/w/api.php"
HEADERS = {"User-Agent": "OgrenciOdevi/1.0 (egitim amacli veri toplama; test@example.com)"}
session = requests.Session()
session.headers.update(HEADERS)

def api_get(params, max_retries=5):
    for attempt in range(max_retries):
        try:
            r = session.get(API, params=params, timeout=15)
            if r.status_code == 429:
                wait = 5 * (attempt + 1)
                print(f"429 alindi, {wait}s bekleniyor...")
                time.sleep(wait)
                continue
            r.raise_for_status()
            return r.json()
        except requests.exceptions.JSONDecodeError:
            time.sleep(3)
        except Exception as e:
            print("GENEL HATA:", repr(e))
            time.sleep(3)
    return None

def get_wikitext_batch(titles):
    data = api_get({
        "action": "query", "format": "json",
        "prop": "revisions", "rvprop": "content", "rvslots": "main",
        "titles": "|".join(titles), "redirects": True,
    })
    if data is None:
        return {}
    pages = data.get("query", {}).get("pages", {})
    result = {}
    for page in pages.values():
        revs = page.get("revisions")
        if revs:
            result[page.get("title")] = revs[0]["slots"]["main"]["*"]
    return result

def clean_wikitext(text):
    text = re.sub(r'\{\|.*?\|\}', '', text, flags=re.S)
    text = re.sub(r'\{\{[^{}]*\}\}', '', text)
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]*)\]\]', r'\1', text)
    text = re.sub(r"'''?", '', text)
    text = re.sub(r'==+\s*([^=]*)\s*==+', r'\1\n', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[\[\]{}|`´±]', '', text)
    text = text.replace('\u00ad', '')
    text = re.sub(r'[\U0001F000-\U0001FFFF]', '', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def extract_links(wikitext):
    links = re.findall(r'\[\[([^\]|#]+)(?:\|[^\]]*)?\]\]', wikitext)
    return [l.strip() for l in links if ':' not in l]

authors = [
    "Kişi:Ömer Seyfettin", "Kişi:Sabahattin Ali", "Kişi:Aziz Nesin",
    "Kişi:Memduh Şevket Esendal", "Kişi:Halid Ziya Uşaklıgil",
    "Kişi:Yaşar Kemal", "Kişi:Ahmet Rasim", "Kişi:Mehmet Rauf",
]

collected = []
total_len = 0
seen = set()
target_chars = 1_000_000

for author in authors:
    if total_len >= target_chars:
        break
    author_data = get_wikitext_batch([author])
    values = list(author_data.values())
    wt = values[0] if values else None
    titles = [t for t in extract_links(wt) if t not in seen]
    print(f"{author}: {len(titles)} link bulundu")

    for i in range(0, len(titles), 50):
        batch = titles[i:i+50]
        seen.update(batch)
        results = get_wikitext_batch(batch)
        for title, raw in results.items():
            cleaned = clean_wikitext(raw)
            if len(cleaned) > 300:
                collected.append(cleaned)
                total_len += len(cleaned)
        time.sleep(1.0)
        if total_len >= target_chars:
            break
    print(f"{author} sonrasi toplam: {total_len} karakter")

turkce_metin = "\n\n".join(collected)[:target_chars]
print("TOPLAM KARAKTER:", len(turkce_metin))
print("FARKLI KARAKTER SAYISI:", len(set(turkce_metin)))

with open("turkce_corpus.txt", "w", encoding="utf-8") as f:
    f.write(turkce_metin)

Kişi:Ömer Seyfettin: 175 link bulundu
Kişi:Ömer Seyfettin sonrasi toplam: 705953 karakter
Kişi:Sabahattin Ali: 9 link bulundu
Kişi:Sabahattin Ali sonrasi toplam: 705953 karakter
Kişi:Aziz Nesin: 1 link bulundu
Kişi:Aziz Nesin sonrasi toplam: 705953 karakter
Kişi:Memduh Şevket Esendal: 3 link bulundu
Kişi:Memduh Şevket Esendal sonrasi toplam: 705953 karakter
Kişi:Halid Ziya Uşaklıgil: 1 link bulundu
Kişi:Halid Ziya Uşaklıgil sonrasi toplam: 705953 karakter
Kişi:Yaşar Kemal: 1 link bulundu
Kişi:Yaşar Kemal sonrasi toplam: 706574 karakter
Kişi:Ahmet Rasim: 1 link bulundu
Kişi:Ahmet Rasim sonrasi toplam: 711130 karakter
Kişi:Mehmet Rauf: 34 link bulundu
Kişi:Mehmet Rauf sonrasi toplam: 711130 karakter
TOPLAM KARAKTER: 711260
FARKLI KARAKTER SAYISI: 101


In [30]:
with open('turkce_corpus.txt', 'r', encoding='utf-8') as f:
    text_tr = f.read()

In [31]:
print("metin uzunluğu:", len(text_tr))

metin uzunluğu: 711260


In [32]:
chars_tr = sorted(list(set(text_tr)))
vocab_size_tr = len(chars_tr)
print("vocab_size:", vocab_size_tr)
print(''.join(chars_tr))

vocab_size: 101

 !"&'()*,-./0123456789:;=?ABCDEFGHIJKLMNOPRSTUVXYZabcdefghijklmnopqrstuvwyz«»ÂÇÖÜâçéîöûüğİıŞş–—‘’“”…


In [33]:
stoi_tr = {ch: i for i, ch in enumerate(chars_tr)}
itos_tr = {i: ch for i, ch in enumerate(chars_tr)}
encode_tr = lambda s: [stoi_tr[c] for c in s]
decode_tr = lambda l: ''.join([itos_tr[i] for i in l])

In [34]:
data_tr = torch.tensor(encode_tr(text_tr), dtype=torch.long)
n_tr = int(0.9 * len(data_tr))
train_data_tr = data_tr[:n_tr]
val_data_tr = data_tr[n_tr:]
print(data_tr.shape, data_tr.dtype)

torch.Size([711260]) torch.int64


In [35]:
def get_batch_tr(split):
    data = train_data_tr if split == 'train' else val_data_tr
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [36]:
@torch.no_grad()
def estimate_loss_tr():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch_tr(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [37]:
vocab_size = vocab_size_tr

model = BigramLanguageModel()
m = model.to(device)
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

2.729573 M parameters


In [38]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [39]:
import time
start_time = time.time()
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss_tr()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    xb, yb = get_batch_tr('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f"Eğitim süresi: {(time.time()-start_time)/60:.2f} dakika")
print(estimate_loss_tr())

step 0: train loss 4.8320, val loss 4.8343
step 500: train loss 2.1703, val loss 2.1075
step 1000: train loss 1.9349, val loss 1.8892
step 1500: train loss 1.7684, val loss 1.7565
step 2000: train loss 1.6490, val loss 1.6713
step 2500: train loss 1.5680, val loss 1.6220
step 3000: train loss 1.5040, val loss 1.5874
step 3500: train loss 1.4569, val loss 1.5595
step 4000: train loss 1.4160, val loss 1.5395
step 4500: train loss 1.3805, val loss 1.5200
step 4999: train loss 1.3533, val loss 1.5118
Eğitim süresi: 10.15 dakika
{'train': tensor(1.3540), 'val': tensor(1.5104)}


In [41]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode_tr(m.generate(context, max_new_tokens=1000)[0].tolist()))


:&mdash; ....
:&mdash; Ne?..
:&mdash; ....
Yain, dar Müşünden ara saldım...
:&mdash; Salevga büyük ağaçlaya berabalarak...
:&mdash; Ya, narge yakabası olan çıkınlar!
:&mdash; Namlar gibi al!
Bu gözlerimi değildir. Suat için kan elçi nasıl ağzı...
:&mdash; ...
:&mdash; Hayır, çorba bir muhal etmek olmasınız...
:&mdash; Allah bu de sabah duramayal ve erseliyiz, böyle kadar zula...
Kendime sarmak istiyordu. Kurudum.
Nazi vardılar, bu mezası bekliyordun? Çocuğun damaıldı meyut elinleti betekli kanıtlarının bir etti. Maısın hetsiz gibi mandın uhut ve İkit değil, balkalıktım. Eşek, bariçer geldim. Yaptırdım. İşte görünmesinin, o muhallii zîm&acirc; ve köpeğemden, kadar teselli gitti:
:&mdash; Köşdürüm.
:&mdash; Boris doğruç, efendir!..
:&mdash; Tabii dapka kaldım. Banla akıyorsunuz bırakmasın... düşündüm.
:&mdash; Solunca döğül bir satlı tutulmuş. Onu turnat Sulayı haraya baktı. Esterdiği Kucağaçla hendisini yerifse ve öz gibi vakitmeden kalma... Laki tarafında. Ve baktı. Yüz kollara iki de

In [ ]:
# Karşılaştırma:
# Tiny Shakespeare, 65 farklı karakter, 0.825M parametre: val loss 1.5585
# Vikikaynak, 101 farklı karakter, 2.729573M parametre:   val loss 1.5104
#
# Hayır, bu iki sayı doğrudan karşılaştırılamaz çünkü:
#
# 1) vocab_size farklı olduğu için  model hiçbir şey öğrenmeden, rastgele tahmin yapsa bile beklenen loss
#    ln(vocab_size) kadar olduğu için Türkçe modelinin başlangıç noktası 101 karakter arasından
#    doğru olanı tahmin ediyor, ingilizce ise 65 karakter arasından tahmin ediyor. Bu yüzden daha zor.
#    Loss'un düşük çıkması bu yüzden otomatik olarak daha iyi öğrendi anlamına gelmiyor.
#
# 2) Model boyutu da farklı. İngilizce sonucu 0.825M parametrelik modelden, Türkçe sonucu
#    2.729573M parametrelik modelden geldi. Yani sadece dil değil, model
#    kapasitesi de değişti iki değişken aynı anda farklı, hangisinin etkisi ne kadar ayıramayız.
#
# 3) Dillerin kendi iç yapıları farklı olduğu için örneğin yüklemin yeri olsun, kelimelerin çok sayıda ek
#    alarak uzaması gibi ingilizce'den farklı karakter/kelime örüntülerine sahip olduğu için, modelin başarısından
#    bağımsız bir özellik.
#
# 4) Veri seti boyutu da tam eşit değil İngilizce 1.11 milyon karakter iken Türkçe 712 bin karakter.